In [40]:
import pandas as pd
from scipy.sparse import hstack, csr_matrix
from sklearn.preprocessing import OneHotEncoder
from sentence_transformers import SentenceTransformer
from catboost import CatBoostClassifier
import pickle

In [2]:
data = pd.read_csv('data/dataset_massive.csv')
data.head()

,Unnamed: 0,published_utc,title,description,publisher,ticker,sentiment
0,0,2026-06-03T20:26:39Z,The 'VOO And Chill' Economy Is Now Worth A His...,Vanguard's S&P 500 ETF (VOO) has become the fi...,Benzinga,AAPL,positive
1,1,2026-06-03T20:26:39Z,The 'VOO And Chill' Economy Is Now Worth A His...,Vanguard's S&P 500 ETF (VOO) has become the fi...,Benzinga,MSFT,positive
2,2,2026-06-03T20:26:39Z,The 'VOO And Chill' Economy Is Now Worth A His...,Vanguard's S&P 500 ETF (VOO) has become the fi...,Benzinga,NVDA,positive
3,3,2026-06-03T19:22:01Z,Broadcom's AI Revenue Doubled — And It May Not...,Broadcom reported $8.4 billion in AI semicondu...,Benzinga,META,positive
4,4,2026-06-03T19:22:01Z,Broadcom's AI Revenue Doubled — And It May Not...,Broadcom reported $8.4 billion in AI semicondu...,Benzinga,AAPL,positive


In [21]:
data_preprocessed = data.copy()

company_names = {"AAPL": "Apple", "MSFT": "Microsoft", "GOOGL": "Google", "META": "Meta", "AMZN": "Amazon", "NVDA": "Nvidia", "NFLX": "Netflix", 
                 "ADBE": "Adobe", "INTC": "Intel", "AMD": "AMD", "IBM": "IBM", "CSCO": "Cisco"}
data_preprocessed["company_name"] = data_preprocessed["ticker"].map(company_names)
data_preprocessed['text'] = (data_preprocessed["title"] + ". " + data_preprocessed["description"])
data_preprocessed['target'] = data_preprocessed['sentiment'].map({'negative': -1, 'neutral': 0, 'positive': 1})
data_preprocessed = data_preprocessed.dropna()

In [24]:
company = data_preprocessed[["company_name"]]
text = data_preprocessed["text"]

ohe_company = OneHotEncoder(handle_unknown="ignore")
X_company = ohe_company.fit_transform(company)

sent_transformer = SentenceTransformer("mukaj/fin-mpnet-base")
X_sentran_text = sent_transformer.encode(text.values.tolist(), batch_size=32, show_progress_bar=True, convert_to_numpy=True)

X_sentran_stacked = hstack([csr_matrix(X_sentran_text), X_company])

y = data_preprocessed[['target']]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/470 [00:00<?, ?it/s]

In [31]:
best_params_catboost = {'iterations': 534, 
                        'learning_rate': 0.08842459522869721, 
                        'depth': 7, 'l2_leaf_reg': 9.371485582972626, 
                        'bagging_temperature': 1.4509350729845583, 
                        'auto_class_weights': 'SqrtBalanced',
                        "loss_function": "MultiClass",
                        "eval_metric": "TotalF1:average=Macro",
                        "random_seed": 42,
                        "task_type": "CPU",
                        "verbose": 0}

best_catboost = CatBoostClassifier(**best_params_catboost)
best_catboost.fit(X_sentran_stacked, y, early_stopping_rounds=100, verbose=0, plot=True)

MetricVisualizer(layout=Layout(align_self='stretch', height='500px'))

CatBoostClassifier(auto_class_weights='SqrtBalanced', bagging_temperature=1.4509350729845583, depth=7, eval_metric='TotalF1:average=Macro', iterations=534, l2_leaf_reg=9.371485582972626, learning_rate=0.08842459522869721, loss_function='MultiClass', random_seed=42, task_type='CPU', verbose=0)

In [43]:
with open("models/model_ohe.pkl", "wb") as f:
    pickle.dump(ohe_company, f)

sent_transformer.save("models/model_sbert")

best_catboost.save_model("models/model_classification.cbm")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]